# Figure 2D — iPSC-EC PCA (9 media conditions, day 0 vs day 40)

Reproduces the manuscript panel: *"UMAP analysis of iPSC-ECs treated with
different viability factors or combinatorial treatments of viability factors
at day 40 shows that VSL restores the transcriptional profile of HAECs at day
40 to that at day 0 (n=3)."*

**Input**: `Fig2D_iPSC_EC_FPKM.csv` — Cuffdiff FPKM, 27 samples = 9 conditions
x 3 biological replicates (GEO libraries S161-S187).

**Pipeline**: the standard bulk RNA-seq exploratory-analysis workflow
described in the Bioconductor RNA-seq workflow (Love, Anders & Huber) and
implemented by DESeq2's `plotPCA()`:

1. log10(FPKM + 1) transform.
2. Keep genes detected (FPKM > 0) in at least 3 samples.
3. Rank genes by variance across samples and keep the top 1000.
4. PCA on that matrix, samples as rows, genes as columns, mean-centered but
   not scaled to unit variance.
5. Plot PC1 vs PC2, coloured by condition group.
6. UMAP on the PCA coordinates (`umap-learn`) as the nonlinear-embedding step.


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
import umap

RNG = 5
OUT = Path("figures"); OUT.mkdir(exist_ok=True)
INPUT = Path("../Fig2D_iPSC_EC_3rep/processed/Fig2D_iPSC_EC_FPKM.csv")

data = pd.read_csv(INPUT, index_col=0)
by_gene = data.set_index("gene_official")
by_gene = by_gene[~by_gene.index.duplicated(keep="first")]
sample_cols = [c for c in by_gene.columns if c != "gene_official"]
expr = by_gene[sample_cols]
print(expr.shape, "genes x", len(sample_cols), "samples")

groups = pd.Series([re.sub(r"_rep\d+$", "", s) for s in sample_cols],
                    index=sample_cols, name="group")
groups.value_counts()

In [ ]:
NTOP = 1000

log_fpkm = np.log10(expr + 1)
detected = (expr > 0).sum(axis=1) >= 3
log_fpkm = log_fpkm[detected]
print(f"{detected.sum()} / {len(detected)} genes detected in >= 3 samples")

top_var = log_fpkm.var(axis=1).sort_values(ascending=False).head(NTOP).index
top500 = log_fpkm.loc[top_var]
print(f"using top {top500.shape[0]} most-variable genes for PCA")

In [ ]:
pca = PCA(n_components=min(10, top500.shape[1] - 1))   # centers, does not scale
pcs = pca.fit_transform(top500.T.values)
pca_df = pd.DataFrame(pcs[:, :2], index=sample_cols, columns=["PC1", "PC2"])
pca_df["group"] = groups
pca_df.to_csv(OUT / "Fig2D_pca_coordinates.csv")

plt.figure(figsize=(4, 3.2))
plt.bar(range(1, len(pca.explained_variance_ratio_) + 1),
        pca.explained_variance_ratio_ * 100)
plt.xlabel("PC"); plt.ylabel("% variance explained")
plt.tight_layout()
plt.savefig(OUT / "Fig2D_pca_variance_ratio.png", dpi=200)
plt.show()

In [ ]:
GROUP_ORDER = ["D0_Ctrl", "D40_Ctrl", "D40_AB", "D40_VEGF", "D40_VSL",
               "D40_VSL_cAMP", "D40_VSL_cGMP", "D40_VSL_cAMP_cGMP",
               "D40_VSL_cAMP_cGMP_AB"]
present = [g for g in GROUP_ORDER if g in pca_df["group"].unique()]

# fixed colour assignment matching the published panel (matplotlib tab10,
# confirmed against the pre-rename UMAP in ppt/figure_name_updated.pptx)
colors = {"D0_Ctrl": "#7f7f7f", "D40_Ctrl": "#bcbd22", "D40_AB": "#1f77b4",
          "D40_VEGF": "#ff7f0e", "D40_VSL": "#2ca02c", "D40_VSL_cAMP": "#d62728",
          "D40_VSL_cGMP": "#e377c2", "D40_VSL_cAMP_cGMP": "#9467bd",
          "D40_VSL_cAMP_cGMP_AB": "#8c564b"}

fig, ax = plt.subplots(figsize=(7, 5.5))
for g in present:
    sub = pca_df[pca_df["group"] == g]
    ax.scatter(sub.PC1, sub.PC2, s=140, color=colors[g], label=g, edgecolor="white", linewidth=0.6)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("iPSC-EC, day 0 vs day 40 (9 conditions) -- PCA")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.tight_layout()
fig.savefig(OUT / "Fig2D_iPSC_EC_PCA.pdf", bbox_inches="tight")
fig.savefig(OUT / "Fig2D_iPSC_EC_PCA.png", dpi=300, bbox_inches="tight")
plt.show()
print("wrote", OUT / "Fig2D_iPSC_EC_PCA.png")

In [ ]:
n_neighbors = 8
min_dist = 0.5
reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, random_state=RNG, init="random")
emb = reducer.fit_transform(pcs)
umap_df = pd.DataFrame(emb, index=sample_cols, columns=["UMAP1", "UMAP2"])
umap_df["group"] = groups
umap_df.to_csv(OUT / "Fig2D_umap_coordinates.csv")

fig, ax = plt.subplots(figsize=(7, 5.5))
for g in present:
    sub = umap_df[umap_df["group"] == g]
    ax.scatter(sub.UMAP1, sub.UMAP2, s=140, color=colors[g], label=g, edgecolor="white", linewidth=0.6)
ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
ax.set_title(f"iPSC-EC -- UMAP on PCA coordinates (n_neighbors={n_neighbors}, random_state={RNG})")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.tight_layout()
fig.savefig(OUT / "Fig2D_iPSC_EC_UMAP.pdf", bbox_inches="tight")
fig.savefig(OUT / "Fig2D_iPSC_EC_UMAP.png", dpi=300, bbox_inches="tight")
plt.show()
print("wrote", OUT / "Fig2D_iPSC_EC_UMAP.png")